# MEV — Máxima Extração de Valor na Rede Polygon
**Pesquisa FAPEMIG** | Coordenador: Prof. José Augusto Miranda Nacif | Bolsista: Aline Cristina Santos Silva

---

Este notebook documenta a **coleta e análise de dados on-chain** da rede Polygon (Layer-2),
com foco na medição do *Reordering Slippage* em swaps da Uniswap V3.

### Objetivo
Coletar eventos de Swap reais da blockchain para, posteriormente, calcular o slippage de reordenação
e comparar com a linha de base da Ethereum Mainnet — validando a hipótese **H** da proposta:
> *Contratos inteligentes em redes L2 apresentam um reordering slippage significativamente menor do que na rede principal Ethereum.*

### Perguntas de Pesquisa endereçadas
- **RQ1:** Qual a magnitude da diferença no Reordering Slippage médio entre Ethereum Mainnet e Polygon?
- **RQ2:** Qual a proporção entre Slippage Adversário (MEV) e Slippage de Colisão (benigno)?
- **RQ3:** Ativos voláteis (memecoins) apresentam maior slippage adversário também em L2?

---

## 1. Instalação de Dependências

Bibliotecas necessárias:
- **`web3`** — interface com nós da blockchain via RPC
- **`pandas`** — manipulação e análise dos dados coletados
- **`python-dotenv`** — carrega variáveis de ambiente do arquivo `.env` (protege a API key)
- **`matplotlib` / `seaborn`** — visualizações

In [1]:
# Execute uma vez para instalar as dependências
%pip install web3 pandas python-dotenv matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Configuração — Carregando Credenciais

A API key da Alchemy fica armazenada no arquivo **`.env`** (nunca versionado no GitHub).
O arquivo `.env.example` no repositório documenta quais variáveis são necessárias sem expor os valores reais.



## 3. Conexão com a Rede Polygon via Alchemy

A conexão é feita via **RPC (Remote Procedure Call)** — um protocolo que permite consultar
o estado da blockchain sem precisar baixar todos os dados localmente.

A **Alchemy** atua como provedor de nó, fornecendo acesso à Polygon Mainnet de forma confiável e escalável.

**Por que Polygon?**
Por ser uma rede L2, espera-se que o *reordering slippage* seja menor do que na Ethereum Mainnet —
essa é exatamente a hipótese que esta pesquisa busca validar empiricamente.

In [3]:
import os
import requests
from dotenv import load_dotenv
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

load_dotenv(override=True)
API_KEY = os.getenv("ALCHEMY_API_KEY")
RPC_URL = f"https://polygon-mainnet.g.alchemy.com/v2/{API_KEY}"

# Sessão customizada sem verificação SSL
session = requests.Session()
session.verify = False

from web3.middleware import ExtraDataToPOAMiddleware
from requests.adapters import HTTPAdapter

w3 = Web3(Web3.HTTPProvider(RPC_URL, session=session))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)

try:
    bloco = w3.eth.block_number
    print(f" Conectado! Bloco: {bloco:,}")
except Exception as e:
    print(f" Erro: {e}")

 Conectado! Bloco: 87,927,286


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 4. Definição do Contrato — Pool Uniswap V3 (USDC/ETH)

Na Uniswap V3, cada par de tokens possui um **contrato de pool dedicado**.
Toda vez que um swap ocorre, o contrato emite um **evento `Swap`** — um log público e imutável
gravado na blockchain, contendo:

| Campo | Descrição | Relevância para a pesquisa |
|---|---|---|
| `sqrtPriceX96` | Preço de execução real codificado | Base para calcular o slippage |
| `amount0 / amount1` | Volume negociado de cada token | Tamanho do swap (impacto no preço) |
| `sender / recipient` | Endereços envolvidos | Identificar padrões de ataque sandwich |
| `tick` | Posição na curva de preço | Faixa de liquidez utilizada |
| `blockNumber` | Bloco em que ocorreu | Agrupar transações por bloco para cálculo do reordering slippage |

Começamos com o pool **USDC/ETH 0.05%** por ser o pool no qual a nossa pesquisa de comparação com a ethereum mainnet se baseia.

In [4]:
# Endereço do pool USDC/WETH 0.05% na Polygon (Uniswap V3)
POOL_ADDRESS = Web3.to_checksum_address("0x45dda9cb7c25131df268515131f647d726f50608")

# ABI mínimo — apenas o evento Swap (não precisamos do ABI completo)
POOL_ABI = [
    {
        "anonymous": False,
        "inputs": [
            {"indexed": True,  "name": "sender",       "type": "address"},
            {"indexed": True,  "name": "recipient",    "type": "address"},
            {"indexed": False, "name": "amount0",      "type": "int256"},
            {"indexed": False, "name": "amount1",      "type": "int256"},
            {"indexed": False, "name": "sqrtPriceX96", "type": "uint160"},
            {"indexed": False, "name": "liquidity",    "type": "uint128"},
            {"indexed": False, "name": "tick",         "type": "int24"}
        ],
        "name": "Swap",
        "type": "event"
    }
]

contrato = w3.eth.contract(address=POOL_ADDRESS, abi=POOL_ABI)
print(f" Contrato carregado: {POOL_ADDRESS}")
print(f" Pool: USDC/ETH 0.05% — Uniswap V3 na Polygon")

 Contrato carregado: 0x45dDa9cb7c25131DF268515131f647d726f50608
 Pool: USDC/ETH 0.05% — Uniswap V3 na Polygon


## 5. Coleta de Eventos de Swap

Estamos usando o pool USDC/ETH (o mesmo pool que foi usado na ethereum mainnet) e estamos usando a janela para 3.000 blocos. 

In [ ]:
import time
import pandas as pd
from pathlib import Path
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.filterwarnings("ignore", category=InsecureRequestWarning)

contrato_wmatic = w3.eth.contract(address=POOL_ADDRESS, abi=POOL_ABI)

JANELA_BLOCOS = 10
TOTAL_BLOCOS  = 15000

# ── Salva/reutiliza sempre o mesmo bloco_fim ──────────────────────────────────
ARQUIVO_BLOCO = Path("dataFrame/polygon_vc_ethereum/bloco_fim.txt")

if ARQUIVO_BLOCO.exists():
    bloco_fim = int(ARQUIVO_BLOCO.read_text().strip())
    print(f" Reutilizando bloco_fim salvo: {bloco_fim:,}")
else:
    bloco_fim = w3.eth.block_number
    ARQUIVO_BLOCO.parent.mkdir(parents=True, exist_ok=True)
    ARQUIVO_BLOCO.write_text(str(bloco_fim))
    print(f" Novo bloco_fim salvo: {bloco_fim:,}")

bloco_inicio = bloco_fim - TOTAL_BLOCOS

print(f"Coletando {TOTAL_BLOCOS:,} blocos em lotes de {JANELA_BLOCOS}...")
print(f"Bloco {bloco_inicio:,} → {bloco_fim:,}")
print()

todos_eventos = []
erros = 0

for i, inicio in enumerate(range(bloco_inicio, bloco_fim, JANELA_BLOCOS)):
    fim = min(inicio + JANELA_BLOCOS - 1, bloco_fim)
    try:
        eventos = contrato_wmatic.events.Swap.get_logs(
            from_block=inicio,
            to_block=fim
        )
        todos_eventos.extend(eventos)
        if i % 50 == 0:
            print(f"  Progresso: bloco {inicio:,} | {len(todos_eventos)} swaps acumulados")
    except Exception as e:
        erros += 1
    time.sleep(0.3)

print(f"\nTotal coletado : {len(todos_eventos)} swaps")
print(f"Erros          : {erros}")

# ── Inspeciona um evento completo ─────────────────────────────────────────────
if todos_eventos:
    print("\n── Exemplo de evento bruto (1º swap coletado) ──")
    ev = todos_eventos[0]
    print(f"  Campos do topo : {list(ev.keys())}")
    print(f"  Args (campos do Swap):")
    for campo, valor in ev["args"].items():
        print(f"    {campo:20s} = {valor}")

# ── Diagnóstico ───────────────────────────────────────────────────────────────
if todos_eventos:
    _tmp = pd.DataFrame([{"bloco": e["blockNumber"]} for e in todos_eventos])
    _por_bloco = _tmp.groupby("bloco").size()
    uteis = (_por_bloco >= 2).sum()
    print(f"\nBlocos com 2+ swaps : {uteis}")
    print(f"Média por bloco     : {_por_bloco.mean():.2f}")
    print(f"Máximo num bloco    : {_por_bloco.max()}")


 Reutilizando bloco_fim salvo: 87,925,372
Coletando 15,000 blocos em lotes de 10...
Bloco 87,910,372 → 87,925,372

  Progresso: bloco 87,910,372 | 2 swaps acumulados
  Progresso: bloco 87,910,872 | 93 swaps acumulados
  Progresso: bloco 87,911,372 | 153 swaps acumulados
  Progresso: bloco 87,911,872 | 238 swaps acumulados
  Progresso: bloco 87,912,372 | 310 swaps acumulados
  Progresso: bloco 87,912,872 | 378 swaps acumulados
  Progresso: bloco 87,913,372 | 454 swaps acumulados
  Progresso: bloco 87,913,872 | 538 swaps acumulados
  Progresso: bloco 87,914,372 | 775 swaps acumulados
  Progresso: bloco 87,914,872 | 890 swaps acumulados
  Progresso: bloco 87,915,372 | 962 swaps acumulados
  Progresso: bloco 87,915,872 | 1011 swaps acumulados
  Progresso: bloco 87,916,372 | 1088 swaps acumulados
  Progresso: bloco 87,916,872 | 1217 swaps acumulados
  Progresso: bloco 87,917,372 | 1330 swaps acumulados
  Progresso: bloco 87,917,872 | 1421 swaps acumulados
  Progresso: bloco 87,918,372 | 155

## 6. Estruturação dos Dados em DataFrame

Convertemos os eventos brutos da blockchain em um **DataFrame pandas** estruturado.
Cada linha representa um swap individual, com todos os campos necessários para
o cálculo do Reordering Slippage na próxima etapa.

In [7]:
import csv
import json
from pathlib import Path

def evento_para_dict(evento):
    """Converte um evento Web3 para um dicionário plano com todos os campos."""
    d = {}

    # ── Campos do topo ────────────────────────────────────────────────────────
    d["event"]            = evento.get("event")
    d["address"]          = evento.get("address")
    d["blockNumber"]      = evento.get("blockNumber")
    d["transactionIndex"] = evento.get("transactionIndex")
    d["logIndex"]         = evento.get("logIndex")

    # HexBytes → string legível
    tx_hash    = evento.get("transactionHash")
    block_hash = evento.get("blockHash")
    d["transactionHash"] = tx_hash.hex()    if tx_hash    else None
    d["blockHash"]       = block_hash.hex() if block_hash else None

    # ── Args (parâmetros do Swap) ─────────────────────────────────────────────
    args = evento.get("args", {})
    for chave, valor in args.items():
        if hasattr(valor, "hex"):                  # HexBytes
            valor = valor.hex()
        elif isinstance(valor, (dict, list)):       # estrutura aninhada
            valor = json.dumps(valor, default=str)
        d[f"args_{chave}"] = valor

    return d


# ── Gera as linhas ────────────────────────────────────────────────────────────
linhas = [evento_para_dict(e) for e in todos_eventos]

if not linhas:
    print("Nenhum evento para salvar.")
else:
    todas_colunas = list(dict.fromkeys(k for linha in linhas for k in linha))

    caminho_csv = Path("dataFrame/polygon_vs_ethereum/swaps.csv")
    with caminho_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=todas_colunas, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(linhas)

    print(f"{len(linhas)} swaps salvos em '{caminho_csv.resolve()}'")
    print(f"   Colunas ({len(todas_colunas)}): {todas_colunas}")

2521 swaps salvos em '/home/aline/slippage-analysis/dataFrame/polygon_vs_ethereum/swaps.csv'
   Colunas (14): ['event', 'address', 'blockNumber', 'transactionIndex', 'logIndex', 'transactionHash', 'blockHash', 'args_sender', 'args_recipient', 'args_amount0', 'args_amount1', 'args_sqrtPriceX96', 'args_liquidity', 'args_tick']


## 7. Conversão do Preço (sqrtPriceX96 → Preço Legível)

O campo `sqrtPriceX96` armazena o preço em formato interno da Uniswap V3:
a **raiz quadrada do preço**, multiplicada por 2⁹⁶ (para evitar decimais na EVM).

Para obter o preço real USDC por ETH , aplicamos:

$$price = \left(\frac{sqrtPriceX96}{2^{96}}\right)^2 \times \frac{10^{decimais\_token0}}{10^{decimais\_token1}}$$

Para USDC (6 decimais) / ETH (18 decimais):

$$price_{USDC/ETH} = \left(\frac{sqrtPriceX96}{2^{96}}\right)^2 \times 10^{12}$$

 Este preço de execução real é a base para calcular o **Reordering Slippage** — a diferença entre o preço que o usuário obteve e o preço que obteria em uma ordem aleatória de transações.

In [15]:
import pandas as pd
from decimal import Decimal

# ── Carrega os dados ──────────────────────────────────────────────────────────
df = pd.read_csv("dataFrame/polygon_vs_ethereum/swaps.csv")

df = df.rename(columns={
    "blockNumber":       "bloco",
    "transactionIndex":  "tx_index",
    "args_sqrtPriceX96": "sqrtPriceX96",
    "args_amount0":      "amount0",
    "args_amount1":      "amount1",
    "args_sender":       "sender",
    "args_recipient":    "recipient",
})

# ── Converte sqrtPriceX96 de string para inteiro SEM passar por float ─────────
# (float64 só tem ~15 dígitos — sqrtPriceX96 tem 24 — perde precisão)
df["sqrtPriceX96"] = df["sqrtPriceX96"].apply(lambda x: int(str(x).strip()))
df["amount0"]      = pd.to_numeric(df["amount0"])
df["amount1"]      = pd.to_numeric(df["amount1"])

# ── Conversão de preço — pool USDC/WETH 0.05% ────────────────────────────────
# token0 = USDC (6 decimais)
# token1 = WETH (18 decimais)
#
# sqrtPriceX96 = sqrt(token1/token0) × 2^96
#
# Passo 1: eleva ao quadrado e divide por 2^96  → ratio bruto (WETH/USDC)
# Passo 2: divide por 10^12                     → corrige decimais (18-6=12)
# Passo 3: inverte (1/x)                        → USDC por WETH

Q96 = Decimal(2 ** 96)

def sqrt_price_to_price(sqrt_price_x96):
    s     = Decimal(int(sqrt_price_x96))
    ratio = (s / Q96) ** 2                      # ratio bruto WETH/USDC
    weth_por_usdc = ratio / Decimal("1e12")     # corrige decimais
    return float(1 / weth_por_usdc)             # inverte → USDC/WETH

df["preco_execucao"] = df["sqrtPriceX96"].apply(sqrt_price_to_price)

# ── Resultado ─────────────────────────────────────────────────────────────────
print("Preços calculados — pool USDC/WETH na Polygon")
print(f"  Preço médio : ${df['preco_execucao'].mean():,.2f} USDC/WETH")
print(f"  Mínimo      : ${df['preco_execucao'].min():,.2f}")
print(f"  Máximo      : ${df['preco_execucao'].max():,.2f}")
print(f"  Total swaps : {len(df)}")
print()

# ── Diagnóstico dos blocos úteis ──────────────────────────────────────────────
swaps_por_bloco = df.groupby("bloco").size()
blocos_uteis    = swaps_por_bloco[swaps_por_bloco >= 2]

print(f"Blocos com 2+ swaps : {len(blocos_uteis)}")
print(f"Swaps nesses blocos : {blocos_uteis.sum()}")
print()

df[["bloco", "tx_index", "preco_execucao", "amount0", "amount1"]].head(10)

Preços calculados — pool USDC/WETH na Polygon
  Preço médio : $1,768.75 USDC/WETH
  Mínimo      : $1,713.13
  Máximo      : $1,792.29
  Total swaps : 2521

Blocos com 2+ swaps : 299
Swaps nesses blocos : 767



,bloco,tx_index,preco_execucao,amount0,amount1
0,87910374,210,1730.976501,1592505,-919546856914408
1,87910376,5,1731.091292,14005717,-8086909237193881
2,87910387,26,1731.098420,869648,-502117487701413
3,87910409,91,1731.417833,38969323,-22497995480367323
4,87910410,116,1731.422234,536924,-309950785792740
5,87910410,124,1731.426747,550506,-317790352975709
6,87910411,48,1731.759162,40551633,-23406976951624847
7,87910411,139,1731.763620,543800,-313858481251742
8,87910412,99,1731.768003,534633,-308566548990890
9,87910415,102,1731.802193,4170685,-2407111024191208


## 8. Cálculo do Reordering Slippage (Adams et al., 2023)

O **Reordering Slippage** compara o preço real de execução de um swap com o preço que ele teria obtido caso os trades do mesmo bloco fossem ordenados aleatoriamente. Se o preço real for pior do que a média das ordens aleatórias, há evidência de reordenação adversarial (MEV/sandwich).

### Fórmula (Definição 3.2)

$$\text{reorderingSlippage}_i = \left(\frac{\text{realizedPrice}_i}{\mathbb{E}_{\pi}[\text{hypotheticalPrice}_i(\pi(S))]} - 1\right) \times -10000$$

onde:
- $\text{realizedPrice}_i$ — preço real de execução do swap $i$
- $\pi(S)$ — permutação aleatória dos trades do bloco $B$
- $\mathbb{E}_{\pi}[\cdot]$ — média sobre `N_PERMS = 200` permutações amostradas

### Simulação do preço hipotético

Como não temos o estado interno do pool a cada bloco, aproximamos o AMM pela invariante $xy = k$: partimos de uma reserva inicial estimada e propagamos os trades anteriores a $i$ (na nova ordem) para descobrir qual seria o preço de execução de $i$ nessa ordenação hipotética.

### Interpretação

| Reordering Slippage | Classificação | Interpretação |
|---|---|---|
| `RS < 0` | **Adversarial** | Usuário pagou mais do que numa ordem aleatória — suspeito de MEV |
| `RS ≥ 0` | **Benigno** | Collision slippage ou price improvement |

In [17]:
import numpy as np
import pandas as pd
from itertools import permutations

# ── Parâmetros ────────────────────────────────────────────────────────────────
N_PERMS = 200          # número de permutações amostradas por bloco
SEED    = 42           # reprodutibilidade
rng     = np.random.default_rng(SEED)

# ── Função auxiliar: simula o preço de execução de um trade em uma ordenação ──

def simular_preco_hipotetico(df_bloco: pd.DataFrame, idx_alvo: int, ordem: list) -> float:
    """
    Simula o preço de execução do trade `idx_alvo` dado que os trades do bloco
    são executados na `ordem` fornecida, partindo do estado inicial do pool
    estimado pelo primeiro trade do bloco.

    Aproximação via AMM xy=k (sem liquidez concentrada).

    Parâmetros
    ----------
    df_bloco : DataFrame com colunas amount0 (WMATIC) e amount1 (USDC)
    idx_alvo : posição LOCAL (0..n-1) do trade de interesse em df_bloco
    ordem    : lista com a permutação das posições locais

    Retorna
    -------
    Preço hipotético em USDC/ETH
    """
    trades = df_bloco[["amount0", "amount1"]].values  # shape (n, 2)

    # Estima o estado INICIAL do pool antes deste bloco:
    # Usamos os amounts do primeiro trade como referência de escala.
    # Ponto de partida: reservas tais que o preço inicial seja o
    # preco_execucao do primeiro trade do bloco.
    preco_inicial = df_bloco["preco_execucao"].iloc[0]

    # Reserva inicial sintética: x0 (ETH), y0 (USDC) com xy = k
    # Escolhemos x0 = |sum amount0| * 10 para ter profundidade razoável
    x0 = abs(trades[:, 0]).sum() * 10   # reserva ETH inicial (unidades brutas)
    y0 = x0 * preco_inicial             # reserva USDC inicial
    k  = x0 * y0                        # invariante

    x, y = x0, y0

    # Executa os trades anteriores ao alvo (na ordem hipotética)
    pos_alvo_na_nova_ordem = ordem.index(idx_alvo)

    for pos in ordem[:pos_alvo_na_nova_ordem]:
        delta0 = trades[pos, 0]   # amount0 (USDC): > 0 se pool recebeu WMATIC
        x_novo = x + delta0
        if x_novo <= 0:
            x_novo = x * 0.001    # proteção numérica
        y = k / x_novo
        x = x_novo

    # Agora executa o trade alvo e mede o preço resultante
    delta0_alvo = trades[idx_alvo, 0]
    x_novo = x + delta0_alvo
    if x_novo <= 0:
        x_novo = x * 0.001

    y_novo = k / x_novo

    delta1_hipotetico = y - y_novo     # USDC que sai do pool (positivo = saiu USDC)

    # Preço = USDC movimentado / ETH movimentado (valor absoluto)
    denom = abs(delta0_alvo)
    if denom == 0:
        return np.nan
    return abs(delta1_hipotetico) / denom


# ── Cálculo principal ─────────────────────────────────────────────────────────

resultados = []

blocos_com_multiplos = df.groupby("bloco").filter(lambda g: len(g) >= 2)["bloco"].unique()

print(f"Calculando reordering slippage para {len(blocos_com_multiplos)} blocos...")
print(f"Permutações por bloco: {N_PERMS}\n")

for i_bloco, bloco_id in enumerate(blocos_com_multiplos):
    df_bloco = df[df["bloco"] == bloco_id].reset_index(drop=True)
    n = len(df_bloco)

    # Para cada trade no bloco, amostramos N_PERMS permutações aleatórias
    for idx_alvo in range(n):
        precos_hipoteticos = []

        for _ in range(N_PERMS):
            ordem = list(rng.permutation(n))
            ph = simular_preco_hipotetico(df_bloco, idx_alvo, ordem)
            if ph is not None and not np.isnan(ph) and ph > 0:
                precos_hipoteticos.append(ph)

        if not precos_hipoteticos:
            continue

        preco_realizado   = df_bloco["preco_execucao"].iloc[idx_alvo]
        preco_hipot_medio = np.mean(precos_hipoteticos)

        # Fórmula do artigo (Adams et al. 2023, Def. 3.2)
        # reorderingSlippage_i = (realizedPrice_i / E[hypotheticalPrice_i] − 1) × −10000
        rs = (preco_realizado / preco_hipot_medio - 1) * -10000

        resultados.append({
            "bloco":               bloco_id,
            "tx_index":            df_bloco["tx_index"].iloc[idx_alvo],
            "preco_realizado":     preco_realizado,
            "preco_hipotetico_medio": preco_hipot_medio,
            "reordering_slippage_bps": rs,
            "n_swaps_no_bloco":    n,
        })

"""    # Progresso
    if (i_bloco + 1) % 10 == 0:
        print(f"  {i_bloco + 1}/{len(blocos_com_multiplos)} blocos processados...")"""

df_rs = pd.DataFrame(resultados)

# ── Resultados ────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"REORDERING SLIPPAGE — Pool USDC/ETH — Polygon")
print(f"{'='*55}")
print(f"Swaps analisados          : {len(df_rs):,}")
print(f"Blocos analisados         : {df_rs['bloco'].nunique():,}")
print()
print(f"Reordering Slippage (bps):")
print(f"  Média                   : {df_rs['reordering_slippage_bps'].mean():.4f}")
print(f"  Mediana                 : {df_rs['reordering_slippage_bps'].median():.4f}")
print(f"  Desvio padrão           : {df_rs['reordering_slippage_bps'].std():.4f}")
print(f"  Mín                     : {df_rs['reordering_slippage_bps'].min():.4f}")
print(f"  Máx                     : {df_rs['reordering_slippage_bps'].max():.4f}")
print()

# Adversarial (RS < 0: usuário pagou mais do que numa ordem aleatória)
adversarial = df_rs[df_rs["reordering_slippage_bps"] < 0]
benign      = df_rs[df_rs["reordering_slippage_bps"] >= 0]
print(f"Adversarial (RS < 0)      : {len(adversarial):,}  ({len(adversarial)/len(df_rs)*100:.1f}%)")
print(f"Benigno    (RS >= 0)      : {len(benign):,}  ({len(benign)/len(df_rs)*100:.1f}%)")
print()
print(f"RS médio adversarial      : {adversarial['reordering_slippage_bps'].mean():.4f} bps")
print(f"RS médio benigno          : {benign['reordering_slippage_bps'].mean():.4f} bps")

# ── Salva resultado ───────────────────────────────────────────────────────────
df_rs.to_csv("dataFrame/polygon_vs_ethereum/reordering_slippage.csv", index=False)
print(f"\nResultados salvos em 'dataFrame/polygon_vs_ethereum/reordering_slippage.csv'")
df_rs.head(10)

Calculando reordering slippage para 299 blocos...
Permutações por bloco: 200


REORDERING SLIPPAGE — Pool USDC/ETH — Polygon
Swaps analisados          : 767
Blocos analisados         : 299

Reordering Slippage (bps):
  Média                   : -11.4279
  Mediana                 : -138.1101
  Desvio padrão           : 783.8331
  Mín                     : -1125.5488
  Máx                     : 1162.9204

Adversarial (RS < 0)      : 434  (56.6%)
Benigno    (RS >= 0)      : 333  (43.4%)

RS médio adversarial      : -625.4943 bps
RS médio benigno          : 788.8868 bps

Resultados salvos em 'dataFrame/polygon_vs_ethereum/reordering_slippage.csv'


,bloco,tx_index,preco_realizado,preco_hipotetico_medio,reordering_slippage_bps,n_swaps_no_bloco
0,87910410,116,1731.422234,1581.654303,-946.906862,2
1,87910410,124,1731.426747,1570.275330,-1026.262168,2
2,87910411,48,1731.759162,1574.353634,-999.810492,2
3,87910411,139,1731.763620,1590.096380,-890.934926,2
4,87910418,29,1731.828357,1557.404770,-1119.963099,2
5,87910418,59,1731.860669,1582.775423,-941.922925,2
6,87910421,43,1732.338979,1574.834015,-1000.136921,2
7,87910421,96,1732.350117,1569.761323,-1035.754873,2
8,87910439,48,1713.133730,1719.375216,36.300896,3
9,87910439,49,1732.455761,1719.502324,-75.332475,3
